# Tutorial: CivicMorph Boulder Colorado Tutorial

## Audience
Urban analysts, planners, and geospatial developers building speculative scenarios for Boulder, Colorado.

## Prerequisites
- A local clone of CivicMorph
- Python 3.11+
- Optional extras for integrations (`graph2city`, `mesa`, `pyrosm`)
- Boulder inputs in `data/boulder/` (or use `--place "Boulder, Colorado"` mode)

## Learning Goals
By the end of this notebook you will be able to:
1. Build Boulder baseline artifacts from PBF, place-name, or polygon selectors.
2. Generate an ensemble and score with revised Mesa simulation options.
3. Compare `abm`, `dla`, `ca`, `network`, and `multi_scale` modes.
4. Tune policy, network, and regional levers and inspect ABM diagnostics.


## Outline

1. Configure Boulder paths and run directory.
2. Build baseline (with selector fallback logic).
3. Generate an ensemble.
4. Run a policy-aware Mesa scoring pass.
5. Compare all Mesa modes.
6. Export top plans and inspect artifacts.
7. Optional Graph2City roundtrip.
8. Exercise: compare policy bundles under fixed mode.


In [ ]:
from pathlib import Path
import json
import pandas as pd

from civicmorph.pipeline import build_baseline, generate_ensemble, score_ensemble, export_top_plans

CITY = "boulder_co"
RUN_DIR = Path("runs") / f"{CITY}_tutorial"
DATA_DIR = Path("data") / "boulder"

OSM_PBF = DATA_DIR / "boulder.osm.pbf"
DEM = DATA_DIR / "boulder_dem.tif"
FLOOD = DATA_DIR / "boulder_flood.tif"
POLYGON = DATA_DIR / "boulder_polygon.geojson"
GRAPH2CITY_SEED = DATA_DIR / "graph2city_seed"

MESA_MODES = ["abm", "dla", "ca", "network", "multi_scale"]
DEFAULT_POLICY = {
    "policy_upzone": 1.2,
    "policy_transit_investment": 1.3,
    "policy_affordable_housing": 1.0,
    "policy_parking_reduction": 1.1,
    "policy_green_protection": 1.2,
    "network_new_links": 6,
    "network_bus_lane_km": 24.0,
    "network_station_infill": 4,
    "regional_growth_boundary": 0.95,
    "regional_conservation_share": 0.30,
}

RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Run directory:", RUN_DIR)
print("OSM exists:", OSM_PBF.exists())
print("Polygon exists:", POLYGON.exists())
print("Supported Mesa modes:", MESA_MODES)


## Step 1: Build baseline for Boulder

This cell prefers local PBF mode; if absent it falls back to place-name mode.
It also demonstrates transit headway assumptions used when GTFS is unavailable.


In [ ]:
baseline_kwargs = {
    "dem": str(DEM) if DEM.exists() else None,
    "flood": str(FLOOD) if FLOOD.exists() else None,
    "project_dir": RUN_DIR,
    "transit_headway_bus": 12.0,
    "transit_headway_tram": 10.0,
    "transit_headway_rail": 8.0,
}

if OSM_PBF.exists():
    baseline = build_baseline(osm_pbf=str(OSM_PBF), **baseline_kwargs)
    selector_used = "osm_pbf"
elif POLYGON.exists():
    baseline = build_baseline(osm_pbf=None, study_area=str(POLYGON), **baseline_kwargs)
    selector_used = "study_area"
else:
    baseline = build_baseline(osm_pbf=None, place_name="Boulder, Colorado", **baseline_kwargs)
    selector_used = "place_name"

print("Baseline selector:", selector_used)
print(json.dumps(baseline.metadata.get("osm", {}), indent=2))


## Step 2: Generate an ensemble

Generate a deterministic ensemble from Boulder baseline artifacts.


In [ ]:
members = generate_ensemble(
    profile_name="optimistic_courtyard_city",
    ensemble_size=20,
    seed=1,
    project_dir=RUN_DIR,
)

print("Members generated:", len(members))
members.head(3)


## Step 3: Score with revised Mesa options

This pass uses `multi_scale` mode and a Boulder-oriented policy/network/regional bundle.
If Mesa is unavailable locally, it falls back to static-only scoring.


In [ ]:
try:
    scores = score_ensemble(
        project_dir=RUN_DIR,
        with_abm=True,
        abm_top=5,
        seed=1,
        abm_mode="multi_scale",
        ca_tessellation="hex",
        **DEFAULT_POLICY,
    )
except Exception as exc:
    print("ABM scoring fallback to static-only:", exc)
    scores = score_ensemble(project_dir=RUN_DIR, with_abm=False, abm_top=5, seed=1)

scores = scores.sort_values("final_with_abm", ascending=False)
scores[[
    "member_id",
    "abm_mode",
    "static_final",
    "abm_penalty",
    "final_with_abm",
    "abm_growth_focus_index",
    "abm_capacity_utilization",
    "abm_network_access_gain",
]].head(10)


## Step 4: Compare Mesa modes on the same Boulder run

Keep policy levers fixed and vary only `abm_mode` to isolate mode behavior.


In [ ]:
mode_rows = []
for mode in MESA_MODES:
    mode_scores = score_ensemble(
        project_dir=RUN_DIR,
        with_abm=True,
        abm_top=5,
        seed=1,
        abm_mode=mode,
        ca_tessellation="hex",
        **DEFAULT_POLICY,
    )
    top = mode_scores.sort_values("final_with_abm", ascending=False).iloc[0]
    mode_rows.append(
        {
            "abm_mode": mode,
            "top_member": int(top["member_id"]),
            "top_final_with_abm": float(top["final_with_abm"]),
            "mean_abm_penalty": float(mode_scores["abm_penalty"].fillna(0).mean()),
            "mean_growth_focus": float(mode_scores["abm_growth_focus_index"].fillna(0).mean()),
            "mean_capacity_utilization": float(mode_scores["abm_capacity_utilization"].fillna(0).mean()),
            "mean_network_access_gain": float(mode_scores["abm_network_access_gain"].fillna(0).mean()),
        }
    )

pd.DataFrame(mode_rows).sort_values("top_final_with_abm", ascending=False)


Interpretation:
- `abm_penalty` captures post-plan regression signals against baseline.
- `abm_growth_focus_index` indicates concentration of growth dynamics.
- `abm_capacity_utilization` tracks intensity/capacity uptake in simulation.
- `abm_network_access_gain` captures access gain from interventions/coupling.


## Step 5: Export top Boulder plans

Exports static and interactive artifacts for the current score table.


In [ ]:
exports = export_top_plans(project_dir=RUN_DIR, top_n=3, graph2city_out=False)
exports


In [ ]:
exports_dir = RUN_DIR / "exports"
summary_path = exports_dir / "ensemble_summary.md"
abm_summary = RUN_DIR / "abm" / "abm_summary.parquet"

print("Summary exists:", summary_path.exists())
if summary_path.exists():
    print(summary_path.read_text()[:600])
print("ABM summary exists:", abm_summary.exists() or abm_summary.with_name(abm_summary.name + ".csv").exists())


## Optional: Graph2City import/export for Boulder

If `data/boulder/graph2city_seed` and `civicmorph[graph2city]` are available:
1. Re-run generation with `seed_source="hybrid"`.
2. Export top plans with `graph2city_out=True`.


In [ ]:
if GRAPH2CITY_SEED.exists():
    try:
        _ = generate_ensemble(
            profile_name="transit_corridor_city",
            ensemble_size=10,
            seed=2,
            project_dir=RUN_DIR,
            seed_source="hybrid",
            graph2city_in=str(GRAPH2CITY_SEED),
        )
        g2c_exports = export_top_plans(project_dir=RUN_DIR, top_n=2, graph2city_out=True)
        print("Graph2City exports ready:", g2c_exports)
    except Exception as exc:
        print("Graph2City path unavailable in this environment:", exc)
else:
    print("Skipping Graph2City step; seed package not found:", GRAPH2CITY_SEED)


## Exercise: Compare policy bundles under fixed mode

Use `abm_mode="multi_scale"` and compare two policy packages.
- Which package increases `abm_network_access_gain`?
- Which package yields the best top-1 `final_with_abm`?


In [ ]:
policy_bundles = {
    "bundle_transit_push": {
        "policy_upzone": 1.1,
        "policy_transit_investment": 1.4,
        "policy_affordable_housing": 1.0,
        "policy_parking_reduction": 1.2,
        "policy_green_protection": 1.0,
        "network_new_links": 7,
        "network_bus_lane_km": 28.0,
        "network_station_infill": 5,
        "regional_growth_boundary": 0.95,
        "regional_conservation_share": 0.25,
    },
    "bundle_green_resilience": {
        "policy_upzone": 1.0,
        "policy_transit_investment": 1.2,
        "policy_affordable_housing": 1.1,
        "policy_parking_reduction": 1.0,
        "policy_green_protection": 1.4,
        "network_new_links": 4,
        "network_bus_lane_km": 18.0,
        "network_station_infill": 3,
        "regional_growth_boundary": 0.90,
        "regional_conservation_share": 0.35,
    },
}

rows = []
for bundle_name, bundle in policy_bundles.items():
    bundle_scores = score_ensemble(
        project_dir=RUN_DIR,
        with_abm=True,
        abm_top=5,
        seed=2,
        abm_mode="multi_scale",
        ca_tessellation="hex",
        **bundle,
    )
    top = bundle_scores.sort_values("final_with_abm", ascending=False).iloc[0]
    rows.append(
        {
            "bundle": bundle_name,
            "top_member": int(top["member_id"]),
            "top_final_with_abm": float(top["final_with_abm"]),
            "mean_abm_penalty": float(bundle_scores["abm_penalty"].fillna(0).mean()),
            "mean_network_gain": float(bundle_scores["abm_network_access_gain"].fillna(0).mean()),
        }
    )

pd.DataFrame(rows).sort_values("top_final_with_abm", ascending=False)


## Pitfalls and Extensions

Common pitfall:
- Comparing mode outputs while changing both mode and policy bundles at the same time.

Fix:
- For mode comparison, keep levers fixed and vary only `abm_mode`.
- For policy comparison, keep `abm_mode` fixed and vary one lever group at a time.

Extension ideas:
- Add neighborhood-specific summaries from `abm_summary.parquet`.
- Build scenario dashboards keyed by `abm_mode` and policy bundle IDs.
- Add calibration notebooks that tune policy multipliers against local planning targets.
